In [1]:
import pandas as pd
import numpy as np
import os

print("=== MODEL 6 — ACTION ===")
print("Dossier courant :", os.getcwd())

print("\nFichiers disponibles dans data :")
for file in os.listdir("../data"):
    print("-", file)

=== MODEL 6 — ACTION ===
Dossier courant : C:\Users\tarek.alsemaan\Desktop\USGS-Earthquake-AI\notebooks

Fichiers disponibles dans data :
- economic_loss_model_2025.csv
- economic_loss_model_v2_2025.csv
- pager_economic_losses_2025.csv
- README.md
- usgs_damage_2025.csv
- usgs_damage_processed_2025.csv
- usgs_earthquakes_2025.csv


# Model 6 — Action

## 1. Exploration des données disponibles

L'objectif de ce modèle est de déterminer une action recommandée à partir des informations disponibles sur un séisme.

Avant de construire le système de décision, nous devons identifier les données disponibles dans le projet et les variables qui peuvent être utilisées.

Nous allons examiner les différents datasets afin de déterminer quelles informations peuvent contribuer à la recommandation d'une action.

In [2]:
data_files = [
    "usgs_earthquakes_2025.csv",
    "usgs_damage_2025.csv",
    "usgs_damage_processed_2025.csv",
    "economic_loss_model_2025.csv",
    "economic_loss_model_v2_2025.csv",
    "pager_economic_losses_2025.csv"
]

for file in data_files:
    path = "../data/" + file

    print("\n" + "=" * 70)
    print(file)
    print("=" * 70)

    df_temp = pd.read_csv(path)

    print("Dimensions :", df_temp.shape)
    print("Colonnes :")
    print(df_temp.columns.tolist())


usgs_earthquakes_2025.csv
Dimensions : (29062, 35)
Colonnes :
['mag', 'place', 'time', 'updated', 'tz', 'url', 'detail', 'felt', 'cdi', 'mmi', 'alert', 'status', 'tsunami', 'sig', 'net', 'code', 'ids', 'sources', 'types', 'nst', 'dmin', 'rms', 'gap', 'magType', 'type', 'title', 'longitude', 'latitude', 'depth', 'event_id', 'date_time', 'year', 'month', 'day', 'hour']

usgs_damage_2025.csv
Dimensions : (29062, 35)
Colonnes :
['mag', 'place', 'time', 'updated', 'tz', 'url', 'detail', 'felt', 'cdi', 'mmi', 'alert', 'status', 'tsunami', 'sig', 'net', 'code', 'ids', 'sources', 'types', 'nst', 'dmin', 'rms', 'gap', 'magType', 'type', 'title', 'longitude', 'latitude', 'depth', 'event_id', 'date_time', 'year', 'month', 'day', 'hour']

usgs_damage_processed_2025.csv
Dimensions : (904, 39)
Colonnes :
['mag', 'place', 'time', 'updated', 'tz', 'url', 'detail', 'felt', 'cdi', 'mmi', 'alert', 'status', 'tsunami', 'sig', 'net', 'code', 'ids', 'sources', 'types', 'nst', 'dmin', 'rms', 'gap', 'magType

## 2. Analyse des variables utiles pour la décision

Le modèle Action doit utiliser des informations permettant de représenter la gravité et les conséquences potentielles d'un séisme.

Les variables particulièrement intéressantes sont :

- le niveau de dommages (`Damage_Level`) ;
- l'intensité ressentie (`mmi` et `cdi`) ;
- l'importance de l'événement (`sig`) ;
- la présence d'un tsunami (`tsunami`) ;
- l'alerte PAGER (`alert`) ;
- la magnitude et la profondeur ;
- les pertes économiques lorsqu'elles sont disponibles.

Nous allons maintenant examiner la distribution et la disponibilité de ces informations avant de définir la variable cible `Action`.

In [3]:
damage_df = pd.read_csv("../data/usgs_damage_processed_2025.csv")
loss_df = pd.read_csv("../data/economic_loss_model_2025.csv")

print("=== DATASET DAMAGE ===")
print("Nombre d'événements :", len(damage_df))

print("\nDistribution de Damage_Level :")
print(damage_df["Damage_Level"].value_counts(dropna=False))

print("\nDistribution de alert :")
print(damage_df["alert"].value_counts(dropna=False))

print("\nValeurs disponibles pour MMI :")
print(damage_df["mmi"].describe())

print("\nValeurs disponibles pour CDI :")
print(damage_df["cdi"].describe())

print("\nDistribution tsunami :")
print(damage_df["tsunami"].value_counts(dropna=False))

print("\nStatistiques SIG :")
print(damage_df["sig"].describe())

print("\n=== PERTES ÉCONOMIQUES ===")
print("Nombre d'événements :", len(loss_df))

print("\nStatistiques economic_loss :")
print(loss_df["economic_loss"].describe())

# Vérifier combien d'événements Damage possèdent également
# une estimation de perte économique.
common_events = damage_df["event_id"].isin(loss_df["event_id"])

print(
    "\nÉvénements Damage avec données économiques :",
    common_events.sum()
)

print(
    "Événements Damage sans données économiques :",
    (~common_events).sum()
)

=== DATASET DAMAGE ===
Nombre d'événements : 904

Distribution de Damage_Level :
Damage_Level
Faible    865
Modéré     25
Sévère     14
Name: count, dtype: int64

Distribution de alert :
alert
green     865
yellow     25
orange      7
red         7
Name: count, dtype: int64

Valeurs disponibles pour MMI :
count    904.000000
mean       3.919323
std        1.869051
min        0.000000
25%        3.454750
50%        4.004000
75%        4.679000
max        9.953000
Name: mmi, dtype: float64

Valeurs disponibles pour CDI :
count    453.000000
mean       4.311921
std        2.009081
min        0.000000
25%        3.100000
50%        4.100000
75%        5.600000
max        9.100000
Name: cdi, dtype: float64

Distribution tsunami :
tsunami
0    797
1    107
Name: count, dtype: int64

Statistiques SIG :
count     904.000000
mean      505.106195
std       228.297585
min       168.000000
25%       441.500000
50%       475.000000
75%       536.000000
max      2910.000000
Name: sig, dtype: float64

## 3. Relation entre l'alerte PAGER et le niveau de dommages

Le dataset contient 904 événements pour lesquels une alerte PAGER est disponible.

Les niveaux de dommages observés sont :

- Faible : 865 événements
- Modéré : 25 événements
- Sévère : 14 événements

Les alertes PAGER disponibles sont :

- green : 865 événements
- yellow : 25 événements
- orange : 7 événements
- red : 7 événements

Avant de construire la variable `Action`, nous devons vérifier la relation entre l'alerte PAGER et `Damage_Level`.

Cette vérification permet d'éviter de créer une nouvelle cible qui serait simplement une copie d'une variable déjà existante.

In [5]:
comparison = pd.crosstab(
    damage_df["alert"],
    damage_df["Damage_Level"],
    margins=True
)

print("Relation entre alert et Damage_Level :")
display(comparison)

Relation entre alert et Damage_Level :


Damage_Level,Faible,Modéré,Sévère,All
alert,,,,
green,865,0,0,865
orange,0,0,7,7
red,0,0,7,7
yellow,0,25,0,25
All,865,25,14,904


## 4. Profil des niveaux de dommages

La comparaison précédente montre que `Damage_Level` correspond directement aux catégories de l'alerte PAGER :

- `green` → Faible
- `yellow` → Modéré
- `orange` ou `red` → Sévère

La variable `alert` ne doit donc pas être utilisée comme variable explicative pour prédire une cible dérivée de `Damage_Level`, car cela introduirait une fuite de données.

Nous allons maintenant analyser les caractéristiques physiques des événements appartenant aux trois niveaux de dommages.

Cette analyse permettra d'observer comment la magnitude, l'intensité, le nombre de signalements, l'indice SIG et le tsunami évoluent selon le niveau de dommages.

In [7]:
profile_columns = [
    "mag",
    "depth",
    "mmi",
    "cdi_imputed",
    "felt_log_imputed",
    "sig",
    "tsunami"
]

damage_profile = (
    damage_df
    .groupby("Damage_Level")[profile_columns]
    .mean()
    .round(2)
)

print("Profil moyen par niveau de dommages :")
display(damage_profile)

Profil moyen par niveau de dommages :


,mag,depth,mmi,cdi_imputed,felt_log_imputed,sig,tsunami
Damage_Level,,,,,,,
Faible,5.38,46.98,3.76,4.07,2.67,476.98,0.11
Modéré,5.89,23.35,7.09,6.71,3.94,770.40,0.16
Sévère,6.66,17.87,8.38,8.12,5.10,1769.36,0.29


In [8]:
print("Magnitude par niveau de dommages")
display(
    damage_df.groupby("Damage_Level")["mag"]
    .agg(["count", "mean", "median", "min", "max"])
    .round(2)
)

print("\nMMI par niveau de dommages")
display(
    damage_df.groupby("Damage_Level")["mmi"]
    .agg(["mean", "median", "min", "max"])
    .round(2)
)

print("\nSIG par niveau de dommages")
display(
    damage_df.groupby("Damage_Level")["sig"]
    .agg(["mean", "median", "min", "max"])
    .round(2)
)

Magnitude par niveau de dommages


,count,mean,median,min,max
Damage_Level,,,,,
Faible,865,5.38,5.50,3.3,7.6
Modéré,25,5.89,5.80,5.1,7.6
Sévère,14,6.66,6.45,5.3,8.8



MMI par niveau de dommages


,mean,median,min,max
Damage_Level,,,,
Faible,3.76,3.96,0.00,8.79
Modéré,7.09,6.94,5.75,8.92
Sévère,8.38,8.42,6.22,9.95



SIG par niveau de dommages


,mean,median,min,max
Damage_Level,,,,
Faible,476.98,466.0,168,1384
Modéré,770.40,693.0,650,1234
Sévère,1769.36,1851.0,1021,2910


## 5. Définition des actions recommandées

Les analyses précédentes montrent qu'une action ne doit pas être déterminée uniquement à partir de la magnitude.

Par exemple, des événements de magnitude élevée peuvent appartenir à différents niveaux de dommages.

Le système Action utilise donc le niveau de dommages estimé comme information principale de décision.

Trois niveaux d'action sont définis :

| Niveau de dommages | Action recommandée |
|---|---|
| Faible | Surveillance |
| Modéré | Évaluation prioritaire |
| Sévère | Intervention prioritaire |

Cette approche transforme les résultats du modèle de dommages en une recommandation opérationnelle simple.

Le modèle Action constitue ainsi une couche de décision située après les modèles prédictifs du projet.

Il ne s'agit pas d'entraîner artificiellement un nouveau algorithme à reproduire une règle créée manuellement, mais de convertir les résultats prédictifs en une décision interprétable.

In [9]:
def recommend_action(damage_level):
    """
    Détermine l'action recommandée à partir
    du niveau de dommages estimé.
    """

    if damage_level == "Faible":
        return "Surveillance"

    elif damage_level == "Modéré":
        return "Évaluation prioritaire"

    elif damage_level == "Sévère":
        return "Intervention prioritaire"

    else:
        return "Action indéterminée"

### Application de la règle de décision

La fonction précédente est maintenant appliquée aux 904 événements disposant d'une information PAGER.

Une nouvelle variable `Action` est créée dans le dataset.

In [10]:
damage_df["Action"] = damage_df["Damage_Level"].apply(recommend_action)

print("Distribution des actions recommandées :")
print(damage_df["Action"].value_counts())

Distribution des actions recommandées :
Action
Surveillance                865
Évaluation prioritaire       25
Intervention prioritaire     14
Name: count, dtype: int64


## 6. Intégration des pertes économiques

Le niveau de dommages fournit une première indication permettant de déterminer une action.

Cependant, deux séismes appartenant au même niveau de dommages peuvent avoir des conséquences économiques très différentes.

Le projet contient également des estimations de pertes économiques provenant de PAGER pour 734 événements.

Nous allons relier ces informations au dataset Action à l'aide de `event_id`.

L'objectif est d'étudier la distribution des pertes économiques pour chaque niveau d'action avant de décider si cette information doit intervenir dans le système de décision final.

In [11]:
# Conserver uniquement les informations économiques nécessaires
economic_data = loss_df[
    ["event_id", "economic_loss"]
].copy()

# Fusion avec le dataset Action
action_df = damage_df.merge(
    economic_data,
    on="event_id",
    how="left"
)

print("Nombre total d'événements :", len(action_df))

print(
    "Avec perte économique disponible :",
    action_df["economic_loss"].notna().sum()
)

print(
    "Sans perte économique disponible :",
    action_df["economic_loss"].isna().sum()
)

Nombre total d'événements : 904
Avec perte économique disponible : 734
Sans perte économique disponible : 170


In [12]:
economic_by_action = (
    action_df
    .dropna(subset=["economic_loss"])
    .groupby("Action")["economic_loss"]
    .agg(
        count="count",
        median="median",
        mean="mean",
        max="max"
    )
    .round(2)
)

economic_by_action["positive_loss"] = (
    action_df
    .dropna(subset=["economic_loss"])
    .assign(
        positive_loss=lambda x: x["economic_loss"] > 0
    )
    .groupby("Action")["positive_loss"]
    .sum()
)

display(economic_by_action)

,count,median,mean,max,positive_loss
Action,,,,,
Intervention prioritaire,14,444034969.5,5.085413e+09,5.299175e+10,14
Surveillance,695,0.0,4.838420e+03,7.598830e+05,70
Évaluation prioritaire,25,7438346.0,1.704506e+07,8.083639e+07,25


## 7. Disponibilité et présence des pertes économiques

Une valeur économique manquante ne signifie pas qu'un séisme n'a causé aucune perte.

Elle signifie simplement qu'aucune estimation économique n'est disponible dans le dataset utilisé.

Il est donc nécessaire de distinguer trois situations :

- perte économique indisponible ;
- perte économique disponible mais égale à zéro ;
- perte économique positive.

Cette distinction évite d'interpréter incorrectement les données manquantes comme une absence de pertes.

In [14]:
positive_loss_distribution = pd.crosstab(
    action_df["Action"],
    action_df["economic_loss"] > 0,
    margins=True
)

positive_loss_distribution.columns = [
    "Perte nulle ou indisponible",
    "Perte positive",
    "Total"
]

display(positive_loss_distribution)

,Perte nulle ou indisponible,Perte positive,Total
Action,,,
Intervention prioritaire,0,14,14
Surveillance,795,70,865
Évaluation prioritaire,0,25,25
All,795,109,904


In [15]:
def economic_status(value):
    if pd.isna(value):
        return "Indisponible"
    elif value == 0:
        return "Nulle"
    else:
        return "Positive"


action_df["Economic_Status"] = (
    action_df["economic_loss"].apply(economic_status)
)

economic_status_table = pd.crosstab(
    action_df["Action"],
    action_df["Economic_Status"],
    margins=True
)

display(economic_status_table)

Economic_Status,Indisponible,Nulle,Positive,All
Action,,,,
Intervention prioritaire,0,0,14,14
Surveillance,170,625,70,865
Évaluation prioritaire,0,0,25,25
All,170,625,109,904


## 8. Analyse du risque de tsunami

La présence d'un indicateur de tsunami peut constituer une information importante pour la décision opérationnelle.

Cependant, elle ne doit pas automatiquement être interprétée comme une preuve de dommages sévères.

Nous allons donc analyser la fréquence de l'indicateur `tsunami` pour chaque niveau d'action.

Cette information pourra être utilisée comme avertissement complémentaire dans le moteur de décision final.

In [16]:
tsunami_table = pd.crosstab(
    action_df["Action"],
    action_df["tsunami"],
    margins=True
)

tsunami_table.columns = [
    "Sans tsunami",
    "Tsunami",
    "Total"
]

display(tsunami_table)

,Sans tsunami,Tsunami,Total
Action,,,
Intervention prioritaire,10,4,14
Surveillance,766,99,865
Évaluation prioritaire,21,4,25
All,797,107,904


## 9. Construction du moteur de décision Action

Les analyses précédentes montrent que plusieurs informations doivent être distinguées.

Le niveau de dommages constitue l'information principale pour déterminer l'action :

- Faible → Surveillance
- Modéré → Évaluation prioritaire
- Sévère → Intervention prioritaire

L'indicateur `tsunami` ne modifie pas automatiquement l'action, car la majorité des événements associés à cet indicateur appartiennent à la catégorie Surveillance.

Il est utilisé comme avertissement complémentaire.

De même, les pertes économiques observées ne sont pas utilisées pour construire directement l'action. Elles constituent une information permettant d'analyser les conséquences des événements.

Le moteur de décision retourne donc :

- une action ;
- un niveau de priorité ;
- un avertissement tsunami éventuel ;
- une justification de la décision.

In [17]:
def action_engine(damage_level, tsunami):
    """
    Moteur de décision du Model 6.

    Parameters
    ----------
    damage_level : str
        Niveau de dommages estimé.
    tsunami : int
        Indicateur tsunami USGS (0 ou 1).

    Returns
    -------
    action : str
        Action recommandée.
    priority : str
        Niveau de priorité.
    warning : str
        Avertissement complémentaire.
    reason : str
        Justification de la décision.
    """

    if damage_level == "Sévère":
        action = "Intervention prioritaire"
        priority = "Haute"
        reason = "Niveau de dommages sévère."

    elif damage_level == "Modéré":
        action = "Évaluation prioritaire"
        priority = "Moyenne"
        reason = "Niveau de dommages modéré."

    elif damage_level == "Faible":
        action = "Surveillance"
        priority = "Faible"
        reason = "Niveau de dommages faible."

    else:
        action = "Action indéterminée"
        priority = "Indéterminée"
        reason = "Niveau de dommages inconnu."

    if tsunami == 1:
        warning = "Avertissement tsunami"
    else:
        warning = "Aucun avertissement tsunami"

    return action, priority, warning, reason

### Application du moteur aux événements

Le moteur est maintenant appliqué aux 904 événements.

Les différentes composantes de la décision sont enregistrées séparément afin de conserver une sortie interprétable.

In [18]:
decision_results = action_df.apply(
    lambda row: action_engine(
        row["Damage_Level"],
        row["tsunami"]
    ),
    axis=1
)

action_df[
    ["Recommended_Action", "Priority", "Warning", "Reason"]
] = pd.DataFrame(
    decision_results.tolist(),
    index=action_df.index
)

display(
    action_df[
        [
            "event_id",
            "mag",
            "Damage_Level",
            "tsunami",
            "Recommended_Action",
            "Priority",
            "Warning",
            "Reason"
        ]
    ].head(10)
)

,event_id,mag,Damage_Level,tsunami,Recommended_Action,Priority,Warning,Reason
0,nc75202001,4.54,Faible,0,Surveillance,Faible,Aucun avertissement tsunami,Niveau de dommages faible.
1,us7000q9fs,5.40,Faible,0,Surveillance,Faible,Aucun avertissement tsunami,Niveau de dommages faible.
2,us7000q9f3,5.20,Faible,0,Surveillance,Faible,Aucun avertissement tsunami,Niveau de dommages faible.
3,us7000q9b5,6.60,Faible,1,Surveillance,Faible,Avertissement tsunami,Niveau de dommages faible.
4,us7000q99p,5.50,Faible,0,Surveillance,Faible,Aucun avertissement tsunami,Niveau de dommages faible.
5,us7000q97d,6.00,Faible,0,Surveillance,Faible,Aucun avertissement tsunami,Niveau de dommages faible.
6,us7000q8tw,6.20,Faible,0,Surveillance,Faible,Aucun avertissement tsunami,Niveau de dommages faible.
7,us7000q8l7,5.40,Faible,0,Surveillance,Faible,Aucun avertissement tsunami,Niveau de dommages faible.
8,pr2025175000,5.70,Faible,0,Surveillance,Faible,Aucun avertissement tsunami,Niveau de dommages faible.
9,us7000q8be,5.50,Faible,0,Surveillance,Faible,Aucun avertissement tsunami,Niveau de dommages faible.


## 10. Vérification du moteur de décision

Avant d'utiliser le moteur Action, nous devons vérifier son comportement sur les différents niveaux de dommages.

Nous sélectionnons des exemples appartenant aux catégories :

- Faible ;
- Modéré ;
- Sévère.

Cette vérification permet de confirmer que le moteur produit l'action, la priorité et l'avertissement attendus.

In [19]:
test_cases = (
    action_df
    .sort_values("Damage_Level")
    .groupby("Damage_Level", group_keys=False)
    .head(3)
)

display(
    test_cases[
        [
            "event_id",
            "mag",
            "mmi",
            "sig",
            "Damage_Level",
            "tsunami",
            "Recommended_Action",
            "Priority",
            "Warning"
        ]
    ]
)

,event_id,mag,mmi,sig,Damage_Level,tsunami,Recommended_Action,Priority,Warning
0,nc75202001,4.54,4.456,364,Faible,0,Surveillance,Faible,Aucun avertissement tsunami
1,us7000q9fs,5.40,4.712,449,Faible,0,Surveillance,Faible,Aucun avertissement tsunami
2,us7000q9f3,5.20,6.342,419,Faible,0,Surveillance,Faible,Aucun avertissement tsunami
19,us6000qkg7,5.60,5.748,755,Modéré,0,Évaluation prioritaire,Moyenne,Aucun avertissement tsunami
37,us6000qixr,6.30,8.272,972,Modéré,0,Évaluation prioritaire,Moyenne,Aucun avertissement tsunami
44,us6000qigd,6.40,7.031,684,Modéré,1,Évaluation prioritaire,Moyenne,Avertissement tsunami
130,us7000pufs,6.20,8.039,1684,Sévère,0,Intervention prioritaire,Haute,Aucun avertissement tsunami
158,us6000q5ps,5.30,8.086,1023,Sévère,0,Intervention prioritaire,Haute,Aucun avertissement tsunami
189,us7000pn9z,6.70,8.978,2105,Sévère,0,Intervention prioritaire,Haute,Aucun avertissement tsunami


## 11. Tests du moteur de décision

Le moteur de décision est testé indépendamment du dataset afin de vérifier son comportement pour toutes les combinaisons possibles de niveau de dommages et d'indicateur tsunami.

Ces tests permettent de vérifier que :

- le niveau Faible produit une action de Surveillance ;
- le niveau Modéré produit une Évaluation prioritaire ;
- le niveau Sévère produit une Intervention prioritaire ;
- l'indicateur tsunami ajoute un avertissement sans modifier automatiquement le niveau d'action.

In [20]:
test_scenarios = [
    ("Faible", 0),
    ("Faible", 1),
    ("Modéré", 0),
    ("Modéré", 1),
    ("Sévère", 0),
    ("Sévère", 1)
]

for damage_level, tsunami in test_scenarios:

    action, priority, warning, reason = action_engine(
        damage_level,
        tsunami
    )

    print("=" * 60)
    print("Damage Level :", damage_level)
    print("Tsunami      :", tsunami)
    print("Action       :", action)
    print("Priority     :", priority)
    print("Warning      :", warning)
    print("Reason       :", reason)

Damage Level : Faible
Tsunami      : 0
Action       : Surveillance
Priority     : Faible
Reason       : Niveau de dommages faible.
Damage Level : Faible
Tsunami      : 1
Action       : Surveillance
Priority     : Faible
Reason       : Niveau de dommages faible.
Damage Level : Modéré
Tsunami      : 0
Action       : Évaluation prioritaire
Priority     : Moyenne
Reason       : Niveau de dommages modéré.
Damage Level : Modéré
Tsunami      : 1
Action       : Évaluation prioritaire
Priority     : Moyenne
Reason       : Niveau de dommages modéré.
Damage Level : Sévère
Tsunami      : 0
Action       : Intervention prioritaire
Priority     : Haute
Reason       : Niveau de dommages sévère.
Damage Level : Sévère
Tsunami      : 1
Action       : Intervention prioritaire
Priority     : Haute
Reason       : Niveau de dommages sévère.


## 12. Validation automatique du moteur

Les scénarios précédents montrent que le moteur produit les résultats attendus.

Nous ajoutons maintenant des tests automatiques avec `assert`.

Ces tests permettent de vérifier que les règles fondamentales du moteur restent correctes si le code est modifié ultérieurement.

Contrairement à une métrique d'accuracy utilisée pour un modèle de Machine Learning, ces tests vérifient ici le fonctionnement d'un système de décision déterministe.

In [21]:
# Test 1 — Faible
action, priority, warning, reason = action_engine("Faible", 0)

assert action == "Surveillance"
assert priority == "Faible"
assert warning == "Aucun avertissement tsunami"


# Test 2 — Faible avec tsunami
action, priority, warning, reason = action_engine("Faible", 1)

assert action == "Surveillance"
assert priority == "Faible"
assert warning == "Avertissement tsunami"


# Test 3 — Modéré
action, priority, warning, reason = action_engine("Modéré", 0)

assert action == "Évaluation prioritaire"
assert priority == "Moyenne"


# Test 4 — Modéré avec tsunami
action, priority, warning, reason = action_engine("Modéré", 1)

assert action == "Évaluation prioritaire"
assert priority == "Moyenne"
assert warning == "Avertissement tsunami"


# Test 5 — Sévère
action, priority, warning, reason = action_engine("Sévère", 0)

assert action == "Intervention prioritaire"
assert priority == "Haute"


# Test 6 — Sévère avec tsunami
action, priority, warning, reason = action_engine("Sévère", 1)

assert action == "Intervention prioritaire"
assert priority == "Haute"
assert warning == "Avertissement tsunami"


print("Tous les tests du moteur Action ont réussi.")

Tous les tests du moteur Action ont réussi.


### Gestion d'un niveau de dommages inconnu

Le moteur doit également être capable de gérer une valeur de niveau de dommages qui ne correspond pas aux catégories attendues.

Dans ce cas, aucune action opérationnelle ne doit être inventée.

Le moteur retourne donc une action et une priorité indéterminées.

In [22]:
action, priority, warning, reason = action_engine(
    "Inconnu",
    0
)

print("Action   :", action)
print("Priority :", priority)
print("Warning  :", warning)
print("Reason   :", reason)

Action   : Action indéterminée
Priority : Indéterminée
Reason   : Niveau de dommages inconnu.


## 13. Préparation du moteur pour l'application

Le moteur Action doit pouvoir être utilisé en dehors du notebook, notamment par l'application Streamlit.

La logique de décision est donc placée dans un module Python indépendant.

Cette séparation permet :

- de réutiliser le même moteur dans plusieurs parties du projet ;
- d'éviter de dupliquer les règles de décision ;
- de faciliter les tests et la maintenance ;
- d'intégrer Model 6 à l'application Streamlit.

In [23]:
"""
Model 6 — Action

Moteur de décision permettant de transformer un niveau
de dommages estimé en une recommandation opérationnelle.

L'indicateur tsunami est utilisé comme avertissement
complémentaire et ne modifie pas automatiquement
le niveau d'action.
"""


def action_engine(damage_level, tsunami):
    """
    Détermine l'action recommandée à partir du niveau
    de dommages et de l'indicateur tsunami.

    Parameters
    ----------
    damage_level : str
        Niveau de dommages :
        Faible, Modéré ou Sévère.

    tsunami : int
        Indicateur tsunami USGS :
        0 = non
        1 = oui

    Returns
    -------
    tuple
        action, priority, warning, reason
    """

    # Détermination de l'action principale
    if damage_level == "Sévère":
        action = "Intervention prioritaire"
        priority = "Haute"
        reason = "Niveau de dommages sévère."

    elif damage_level == "Modéré":
        action = "Évaluation prioritaire"
        priority = "Moyenne"
        reason = "Niveau de dommages modéré."

    elif damage_level == "Faible":
        action = "Surveillance"
        priority = "Faible"
        reason = "Niveau de dommages faible."

    else:
        action = "Action indéterminée"
        priority = "Indéterminée"
        reason = "Niveau de dommages inconnu."

    # Avertissement complémentaire
    if tsunami == 1:
        warning = "Avertissement tsunami"
    else:
        warning = "Aucun avertissement tsunami"

    return action, priority, warning, reason

## 14. Test du module Action

Le moteur de décision a été placé dans un module Python indépendant.

Nous vérifions maintenant que ce module peut être chargé et utilisé directement depuis le notebook.

In [31]:
import sys
from pathlib import Path
import importlib

project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import models.action_engine as ae

importlib.reload(ae)

print("Fichier chargé :")
print(ae.__file__)

print("\nContenu reconnu par Python :")
print(dir(ae))

Fichier chargé :
C:\Users\tarek.alsemaan\Desktop\USGS-Earthquake-AI\models\action_engine.py

Contenu reconnu par Python :
['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'action_engine']


### Vérification du module externe

Le module `action_engine.py` est maintenant correctement détecté par Python.

Nous vérifions son fonctionnement avec un scénario de dommages sévères associé à un indicateur tsunami.

In [32]:
result = ae.action_engine(
    damage_level="Sévère",
    tsunami=1
)

print("Action   :", result[0])
print("Priority :", result[1])
print("Warning  :", result[2])
print("Reason   :", result[3])

Action   : Intervention prioritaire
Priority : Haute
Reason   : Niveau de dommages sévère.


## 15. Validation du moteur sur l'ensemble des événements

Le module externe `action_engine.py` fonctionne correctement sur un scénario individuel.

Nous allons maintenant appliquer le moteur aux 904 événements du dataset.

Pour chaque événement, le moteur génère :

- une action recommandée ;
- un niveau de priorité ;
- un avertissement tsunami éventuel ;
- une justification de la décision.

Cette étape permet de vérifier le comportement du moteur de décision sur l'ensemble des événements disponibles.

In [35]:
production_results = action_df.apply(
    lambda row: ae.action_engine(
        row["Damage_Level"],
        row["tsunami"]
    ),
    axis=1
)

action_df[
    [
        "Production_Action",
        "Production_Priority",
        "Production_Warning",
        "Production_Reason"
    ]
] = pd.DataFrame(
    production_results.tolist(),
    index=action_df.index
)

print("Nombre d'événements :", len(action_df))

print("\nActions :")
print(action_df["Production_Action"].value_counts())

print("\nPriorités :")
print(action_df["Production_Priority"].value_counts())

print("\nAvertissements :")
print(action_df["Production_Warning"].value_counts())

Nombre d'événements : 904

Actions :
Production_Action
Surveillance                865
Évaluation prioritaire       25
Intervention prioritaire     14
Name: count, dtype: int64

Priorités :
Production_Priority
Faible     865
Moyenne     25
Haute       14
Name: count, dtype: int64

Avertissements :
Production_Warning
Aucun avertissement tsunami    797
Avertissement tsunami          107
Name: count, dtype: int64


## 16. Validation automatique du moteur de décision

Après avoir appliqué le moteur aux 904 événements, nous effectuons plusieurs contrôles automatiques.

Ces tests permettent de vérifier que :

- chaque événement possède une action ;
- chaque événement possède une priorité ;
- aucune action n'est indéterminée ;
- les niveaux de dommages sont correctement transformés en actions ;
- l'avertissement tsunami correspond exactement à l'indicateur USGS.

Ces contrôles vérifient la cohérence du moteur de décision.

Ils ne constituent pas une mesure d'accuracy d'un modèle de Machine Learning, car les actions sont déterminées par des règles explicites.

In [36]:
# 1. Vérifier les valeurs manquantes

assert action_df["Production_Action"].isna().sum() == 0
assert action_df["Production_Priority"].isna().sum() == 0
assert action_df["Production_Warning"].isna().sum() == 0
assert action_df["Production_Reason"].isna().sum() == 0


# 2. Vérifier qu'aucune action n'est indéterminée

assert (
    action_df["Production_Action"] != "Action indéterminée"
).all()


# 3. Vérifier les règles de décision

assert (
    action_df.loc[
        action_df["Damage_Level"] == "Faible",
        "Production_Action"
    ] == "Surveillance"
).all()

assert (
    action_df.loc[
        action_df["Damage_Level"] == "Modéré",
        "Production_Action"
    ] == "Évaluation prioritaire"
).all()

assert (
    action_df.loc[
        action_df["Damage_Level"] == "Sévère",
        "Production_Action"
    ] == "Intervention prioritaire"
).all()


# 4. Vérifier les priorités

assert (
    action_df.loc[
        action_df["Damage_Level"] == "Faible",
        "Production_Priority"
    ] == "Faible"
).all()

assert (
    action_df.loc[
        action_df["Damage_Level"] == "Modéré",
        "Production_Priority"
    ] == "Moyenne"
).all()

assert (
    action_df.loc[
        action_df["Damage_Level"] == "Sévère",
        "Production_Priority"
    ] == "Haute"
).all()


# 5. Vérifier les avertissements tsunami

assert (
    action_df["Production_Warning"].eq(
        "Avertissement tsunami"
    )
    ==
    action_df["tsunami"].eq(1)
).all()


print("Validation réussie sur les", len(action_df), "événements.")
print("Aucune valeur manquante.")
print("Aucune action indéterminée.")
print("Aucune incohérence détectée.")

Validation réussie sur les 904 événements.
Aucune valeur manquante.
Aucune action indéterminée.
Aucune incohérence détectée.


## 17. Exemples de décisions produites

Afin d'illustrer le fonctionnement du moteur, nous sélectionnons plusieurs événements représentant les trois niveaux de dommages.

Pour chaque séisme, nous affichons :

- la magnitude ;
- l'intensité MMI ;
- l'indice de significativité `sig` ;
- le niveau de dommages ;
- l'indicateur tsunami ;
- l'action recommandée ;
- la priorité ;
- l'avertissement éventuel.

Ces exemples permettent d'observer comment les résultats des modèles sont transformés en recommandations opérationnelles.

In [37]:
examples = (
    action_df
    .groupby("Damage_Level", group_keys=False)
    .head(3)
)

display(
    examples[
        [
            "event_id",
            "mag",
            "mmi",
            "sig",
            "Damage_Level",
            "tsunami",
            "Production_Action",
            "Production_Priority",
            "Production_Warning"
        ]
    ]
)

,event_id,mag,mmi,sig,Damage_Level,tsunami,Production_Action,Production_Priority,Production_Warning
0,nc75202001,4.54,4.456,364,Faible,0,Surveillance,Faible,Aucun avertissement tsunami
1,us7000q9fs,5.40,4.712,449,Faible,0,Surveillance,Faible,Aucun avertissement tsunami
2,us7000q9f3,5.20,6.342,419,Faible,0,Surveillance,Faible,Aucun avertissement tsunami
19,us6000qkg7,5.60,5.748,755,Modéré,0,Évaluation prioritaire,Moyenne,Aucun avertissement tsunami
37,us6000qixr,6.30,8.272,972,Modéré,0,Évaluation prioritaire,Moyenne,Aucun avertissement tsunami
44,us6000qigd,6.40,7.031,684,Modéré,1,Évaluation prioritaire,Moyenne,Avertissement tsunami
130,us7000pufs,6.20,8.039,1684,Sévère,0,Intervention prioritaire,Haute,Aucun avertissement tsunami
158,us6000q5ps,5.30,8.086,1023,Sévère,0,Intervention prioritaire,Haute,Aucun avertissement tsunami
189,us7000pn9z,6.70,8.978,2105,Sévère,0,Intervention prioritaire,Haute,Aucun avertissement tsunami


## 20. Sauvegarde des résultats

Les résultats du moteur Action sont sauvegardés afin de pouvoir être réutilisés dans les autres composants du projet.

Le fichier contient les informations principales décrivant le séisme ainsi que la décision produite par le moteur.

Cette sortie peut notamment être utilisée pour l'analyse, les démonstrations et l'application Streamlit.

In [38]:
output_columns = [
    "event_id",
    "place",
    "mag",
    "depth",
    "mmi",
    "cdi",
    "sig",
    "tsunami",
    "Damage_Level",
    "Production_Action",
    "Production_Priority",
    "Production_Warning",
    "Production_Reason"
]

action_output = action_df[output_columns].copy()

output_path = "../data/action_model_2025.csv"

action_output.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("Fichier sauvegardé :", output_path)
print("Dimensions :", action_output.shape)

display(action_output.head())

Fichier sauvegardé : ../data/action_model_2025.csv
Dimensions : (904, 13)


,event_id,place,mag,depth,mmi,cdi,sig,tsunami,Damage_Level,Production_Action,Production_Priority,Production_Warning,Production_Reason
0,nc75202001,"8 km SSE of Deep Springs, CA",4.54,11.6,4.456,3.4,364,0,Faible,Surveillance,Faible,Aucun avertissement tsunami,Niveau de dommages faible.
1,us7000q9fs,"98 km NNW of Tatsugō, Japan",5.40,10.0,4.712,NaN,449,0,Faible,Surveillance,Faible,Aucun avertissement tsunami,Niveau de dommages faible.
2,us7000q9f3,"56 km NNE of Barkhan, Pakistan",5.20,10.0,6.342,8.6,419,0,Faible,Surveillance,Faible,Aucun avertissement tsunami,Niveau de dommages faible.
3,us7000q9b5,Scotia Sea,6.60,10.0,0.000,NaN,670,1,Faible,Surveillance,Faible,Avertissement tsunami,Niveau de dommages faible.
4,us7000q99p,"161 km NNW of Houma, Tonga",5.50,229.0,3.002,NaN,465,0,Faible,Surveillance,Faible,Aucun avertissement tsunami,Niveau de dommages faible.


## 21. Vérification du fichier exporté

Le fichier produit par le Model 6 est relu afin de vérifier son intégrité.

Nous contrôlons :

- le nombre de lignes et de colonnes ;
- l'absence de doublons dans `event_id` ;
- l'absence d'actions manquantes ;
- la distribution finale des actions.

Cette vérification garantit que le fichier sauvegardé peut être réutilisé par les autres composants du projet.

In [39]:
verification_df = pd.read_csv(
    "../data/action_model_2025.csv"
)

print("Dimensions :", verification_df.shape)

print(
    "Doublons event_id :",
    verification_df["event_id"].duplicated().sum()
)

print(
    "Actions manquantes :",
    verification_df["Production_Action"].isna().sum()
)

print(
    "Priorités manquantes :",
    verification_df["Production_Priority"].isna().sum()
)

print("\nDistribution finale des actions :")

print(
    verification_df[
        "Production_Action"
    ].value_counts()
)

Dimensions : (904, 13)
Doublons event_id : 0
Actions manquantes : 0
Priorités manquantes : 0

Distribution finale des actions :
Production_Action
Surveillance                865
Évaluation prioritaire       25
Intervention prioritaire     14
Name: count, dtype: int64


# Conclusion générale — Model 6 Action

Le Model 6 transforme les résultats du système d'analyse sismique en recommandations opérationnelles interprétables.

Le moteur utilise le niveau de dommages comme signal principal de décision :

| Niveau de dommages | Action recommandée | Priorité |
|---|---|---|
| Faible | Surveillance | Faible |
| Modéré | Évaluation prioritaire | Moyenne |
| Sévère | Intervention prioritaire | Haute |

L'indicateur tsunami est utilisé comme avertissement complémentaire sans provoquer automatiquement une escalade de l'action.

Cette décision est justifiée par l'analyse des données : un indicateur tsunami n'implique pas nécessairement un niveau de dommages sévère.

Le moteur a été séparé du notebook dans le module :

`models/action_engine.py`

Il peut ainsi être réutilisé directement par l'application Streamlit et les autres composants du projet.

## Résultats finaux

- Événements analysés : **904**
- Surveillance : **865**
- Évaluation prioritaire : **25**
- Intervention prioritaire : **14**
- Avertissements tsunami : **107**
- Actions manquantes : **0**
- Actions indéterminées : **0**
- Doublons `event_id` : **0**

Les résultats ont été exportés dans :

`data/action_model_2025.csv`

Le Model 6 est un moteur de décision déterministe et non un modèle supervisé entraîné sur une cible indépendante.

Par conséquent, aucune métrique artificielle d'accuracy, de précision ou de F1-score n'est présentée pour ce modèle.

Les tests réalisés vérifient plutôt la cohérence, l'intégrité et la reproductibilité des règles de décision.